In [32]:
import textwrap
import sys 
sys.path.append('../..')
import utils.utils as utils


with open('../../data/arxiv/cleaned_DPO.txt', 'r') as f:
    paper = f.read()


def print_wrapped(text, width=100):
    """
    Prints the given text wrapped to a specified width for better readability in notebooks.
    This function preserves paragraph breaks.
    """
    paragraphs = text.split('\n\n')
    for para in paragraphs:
        print(textwrap.fill(para, width=width))
        print()
        
print_wrapped(paper)

\title{Direct Preference Optimization: Your Language Model is Secretly a Reward Model}

\begin{abstract} While large-scale unsupervised language models (LMs) learn broad world knowledge
and some reasoning skills, achieving precise control of their behavior is difficult due to the
completely unsupervised nature of their training. Existing methods for gaining such steerability
collect human labels of the relative quality of model generations and fine-tune the unsupervised LM
to align with these preferences, often with reinforcement learning from human feedback (RLHF).
However, RLHF is a complex and often unstable procedure, first fitting a reward model that reflects
the human preferences, and then fine-tuning the large unsupervised LM using reinforcement learning
to maximize this estimated reward without drifting too far from the original model. In this paper we
introduce a new parameterization of the reward model in RLHF that enables extraction of the
corresponding optimal policy in clo

### [Old Prompts] Generate Factual Recall Knowledge Probe

In [6]:
prompt = {}
# prompt['system'] = """Your task is to extract the key knowledge facts in a paper. 

# ##
# These claims should be stand alone claims that have meaning on their own without context, offer generalizable knowledge, and are descriptive rather than prescriptive or evaluative e.g. . For instance, the claim shouldn't be specific to a particular experiment of the paper, but a general insight that follows. List them word for word how it appears in the paper."""
# prompt['system'] = """Your task is to act as a knowledge extraction engine. From the provided text, extract standalone, factual claims that represent generalizable knowledge. The claims should be objective, descriptive, and self-contained.
# # Key Criteria For Extracting Claims

# 1.  **Objective and Factual:** Extract statements presented as facts or claims, not opinions, subjective evaluations, or superlative statements.
#     *   GOOD: "Direct Preference Optimization (DPO) is an algorithm for aligning language models with human preferences."
#     *   BAD: "We believe DPO is the most promising approach for alignment." (Subjective belief)
#     *   BAD: "DPO is the best method." (Superlative statement)

# 2.  **Self-Contained:** The claim must be fully understandable on its own, without requiring context from surrounding sentences. It should not include any pronouns or references that aren't included inthe claim.
#     *   GOOD: "The DPO loss function is derived from a mapping between policy probabilities and an implicit reward."
#     *   BAD: "This loss function is more stable." (Requires external context for "This")

# 3.  **Generalizable Knowledge, not Paper-Specific Results:** Focus on definitions, mechanisms, and core concepts. Do not extract claims about the paper's specific findings, experimental setup, or citations.
#     *   GOOD: "Reinforcement Learning from Human Feedback (RLHF) typically involves training a separate reward model on preference data."
#     *   BAD: "Our experiments show a 5% improvement on the benchmark." (Specific result)
#     *   BAD: "As shown by Smith et al. (2023), the method is effective." (Relies on a citation)
prompt['system'] = """Carefully read the provided text. Your job is to extract knowledge from the provided paper.

# Detailed Instructions
 - Specifically, I want you to identify the "pieces of knowledge" from the following text.
    - We define a piece of knowledge as an informative statement that makes sense on its own without further context. 
    - The piece of knowledge can be a single sentence, or a paragraph, or even a section of the paper. Whatever the case, it should be fully self-contained and coherent. 
    - Demarcate the piece of knowledge by marking the beginning and end of the "knowledge" in the paper with <knowledge> and </knowledge> tags.
    - Again, place the tags around the entire piece of knowledge so that it's self-contained. For instance, if there is a reference to a "reward function", the earlier formal mention of the reward function should be included in the span of the piee of knowledge.
 - Do this for all pieces of knowledge from the following text.
 - Return the entire text with this annotation."""

# 2.  **Descriptive, not Prescriptive:** The claim should describe what something *is* or *does*, not give advice or instructions.
#     *   GOOD: "DPO directly optimizes the language model policy using preference data."
#     *   BAD: "To align your model, you should use the DPO method." (Prescriptive advice)

prompt['user'] = f"""Paper: {paper}"""
extracted_claims = []
import concurrent.futures

def query_single():
    return utils.query_llm(prompt, model='gpt-4.1', temperature=0.1, max_tokens=32768)

with concurrent.futures.ThreadPoolExecutor() as executor:
    futures = [executor.submit(query_single) for _ in range(1)]
    extracted_claims = [future.result() for future in futures]
print(extracted_claims[0])

Paper: \title{Direct Preference Optimization: Your Language Model is Secretly a Reward Model}

\begin{abstract}
<knowledge>
While large-scale unsupervised language models (LMs) learn broad world knowledge and some reasoning skills, achieving precise control of their behavior is difficult due to the completely unsupervised nature of their training.
Existing methods for gaining such steerability collect human labels of the relative quality of model generations and fine-tune the unsupervised LM to align with these preferences, often with reinforcement learning from human feedback (RLHF).
However, RLHF is a complex and often unstable procedure, first fitting a reward model that reflects the human preferences, and then fine-tuning the large unsupervised LM using reinforcement learning to maximize this estimated reward without drifting too far from the original model.
In this paper we introduce a new parameterization of the reward model in RLHF that enables extraction of the corresponding op

In [20]:
extraction_prompt = """Your task is to act as a text segmenter. Carefully read the provided text from a research paper and identify all "pieces of knowledge."

# Definition of a "Piece of Knowledge"
A piece of knowledge is an informative statement that is:
1.  **Self-Contained:** It must be fully understandable on its own without needing sentences before or after it. If it uses a specific term (e.g., "the reward function"), the definition or first mention of that term should be included in the segment.
2.  **Coherent:** The extracted text should form a logical, complete thought. This can range from a single, dense sentence to a full paragraph that explains a concept.

# What to Exclude (Do NOT tag these):
- **Paper Structure & Meta-Commentary:** Sentences about the paper itself (e.g., "In this section, we describe...", "Figure 3 shows...", "Our contributions are threefold..."). 
- **Citations:** Sentences that primarily exist to cite other work (e.g., "This was previously shown by Smith et al. (2023).").
- **Transitional Sentences:** Text that only serves to connect ideas without adding new information (e.g., "Furthermore...", "Now we turn to...", "In addition to the above...").
- **Author Speculation or Rhetorical Questions:** Subjective statements or questions posed to the reader (e.g., "This result is quite surprising.", "But what if the model could...?").

# Instructions
1.  Read the entire text carefully.
2.  Identify all segments that fit the definition of a "piece of knowledge" and do not fall into the exclusion categories.
3.  Demarcate each piece of knowledge by wrapping it in `<knowledge>` and `</knowledge>` tags.
4.  Return the entire original text with these annotations. Do not modify or summarize the text itself.
"""
prompt['user'] = f"""Paper: {paper}"""
extracted_claims = []
import concurrent.futures

def query_single():
    return utils.query_llm(prompt, model='gpt-4.1', temperature=0.1, max_tokens=32768)

with concurrent.futures.ThreadPoolExecutor() as executor:
    futures = [executor.submit(query_single) for _ in range(1)]
    extracted_claims = [future.result() for future in futures]
print_wrapped(extracted_claims[0])

Paper: \title{Direct Preference Optimization: Your Language Model is Secretly a Reward Model}

\begin{abstract} <knowledge> While large-scale unsupervised language models (LMs) learn broad world
knowledge and some reasoning skills, achieving precise control of their behavior is difficult due to
the completely unsupervised nature of their training. Existing methods for gaining such steerability
collect human labels of the relative quality of model generations and fine-tune the unsupervised LM
to align with these preferences, often with reinforcement learning from human feedback (RLHF).
However, RLHF is a complex and often unstable procedure, first fitting a reward model that reflects
the human preferences, and then fine-tuning the large unsupervised LM using reinforcement learning
to maximize this estimated reward without drifting too far from the original model. In this paper we
introduce a new parameterization of the reward model in RLHF that enables extraction of the
corresponding op

In [21]:
extraction_prompt = """Your task is to act as a text segmenter. Carefully read the provided text from a paper and identify all "pieces of knowledge."

# Definition of a "Piece of Knowledge"
A piece of knowledge is an informative statement that is:
1.  Self-Contained: It must be fully understandable on its own without needing sentences before or after it. 
    - If it's referring to a term in a previous sentence (e.g., "the reward function"), they should be together as one unit of knowledge.
    - If the sentence is directly building on a previous sentence, either by a pronoun or a reference, they should be together as one unit of knowledge.
2.  Atomic and Coherent: It should be a single, complete thought.  
    - This can range from a single, dense sentence to a full paragraph that explains it.

# What to Exclude (Do NOT tag these):
Generally speaking, exclude sentences that mostly consist of common words and phrases that don't add much information:
- Paper Structure & Meta-Commentary: Sentences about the paper itself (e.g., "In this section, we describe...", "Figure 3 shows...", "Our contributions are threefold..."). 
- Transitional Sentences: Text that only serves to connect ideas without adding new information (e.g., "Furthermore...", "Now we turn to...", "In addition to the above...").
- Author Speculation or Rhetorical Questions: Subjective statements or questions posed to the reader (e.g., "This result is quite surprising.", "But what if the model could...?"). 
- Figures and Tables: Sentences that describe figures and tables.

# Instructions
1.  Read the entire text carefully.
2.  Identify all segments that fit the definition of a "piece of knowledge" and do not fall into the exclusion categories.
3.  Demarcate each piece of knowledge by wrapping it in `<knowledge>` and `</knowledge>` tags.
4.  While you should ensure the knowledge is self-contained and atomic as much as you can, do not just place tags around an entire paragraph. Encapsulate smaller, discrete units of knowledge. When appropriate, you can also have nested tags.
5.  Return the entire original text with these annotations. Do not modify or summarize the text itself.

# Example
This is an example in which nested tags are used because the first sentence is self-contained and atomic, but the follow-up sentence is not and thus it relies on the previous sentence.
<knowledge><knowledge>In this paper we introduce a new parameterization of the reward model in RLHF that enables extraction of the corresponding optimal policy in closed form, allowing us to solve the standard RLHF problem with only a simple classification loss.</knowledge> The resulting algorithm, which we call \textit{Direct Preference Optimization} (DPO), is stable, performant, and computationally lightweight, eliminating the need for sampling from the LM during fine-tuning or performing significant hyperparameter tuning.</knowledge>
"""
# - For instance, it's okay if you have to forgo some portions of a paragraph to break it up into smaller units.
# - Citations: Sentences that primarily exist to cite other work (e.g., "This was previously shown by Smith et al. (2023).").


# Split paper into paragraphs
paper_paragraphs = paper.split('\n\n')

extracted_claims = []
import concurrent.futures

def query_single(paragraph):
    prompt = {}
    prompt['system'] = extraction_prompt
    prompt['user'] = f"""Paper: {paragraph}"""
    return utils.query_llm(prompt, model='gpt-4.1', temperature=0, max_tokens=32768)

with concurrent.futures.ThreadPoolExecutor() as executor:
    futures = [executor.submit(query_single, paragraph) for paragraph in paper_paragraphs]
    extracted_claims = [future.result() for future in futures]

# Print each output
for i, output in enumerate(extracted_claims, 1):
    print_wrapped(f"Paragraph {i}: {output}")
    print_wrapped("-" * 50)

Paragraph 1: Paper: \title{Direct Preference Optimization: Your Language Model is Secretly a Reward
Model}

--------------------------------------------------

Paragraph 2: \begin{abstract} <knowledge>While large-scale unsupervised language models (LMs) learn
broad world knowledge and some reasoning skills, achieving precise control of their behavior is
difficult due to the completely unsupervised nature of their training.</knowledge>
<knowledge>Existing methods for gaining such steerability collect human labels of the relative
quality of model generations and fine-tune the unsupervised LM to align with these preferences,
often with reinforcement learning from human feedback (RLHF).</knowledge> <knowledge>However, RLHF
is a complex and often unstable procedure, first fitting a reward model that reflects the human
preferences, and then fine-tuning the large unsupervised LM using reinforcement learning to maximize
this estimated reward without drifting too far from the original model.</k

In [25]:
extraction_prompt = """Your task is to act as a text segmenter. Carefully read the provided text from a paper and identify all "pieces of knowledge."

# Definition of a "Piece of Knowledge"
A piece of knowledge is statement that:
1. Provides information.
2. Self-Contained
    - It must be fully understandable on its own without needing sentences before or after it. 
    - If it's referring to a term in a previous sentence (e.g., "the reward function"), they should be together as one unit of knowledge.
    - If the sentence is directly building on a previous sentence, either by a pronoun or a reference, they should be together as one unit of knowledge.
2. Atomic and Coherent
    - It should be a single, complete thought.  
    - This can range from a single, dense phrase to a few sentences that are woven together.

# What to Exclude (Do NOT tag these):
Generally speaking, exclude sentences that mostly consist of common words and phrases that don't add much information:
- Paper Structure & Meta-Commentary: Sentences about the paper itself (e.g., "In this section, we describe...", "Figure 3 shows...", "Our contributions are threefold..."). 
- Transitional Sentences: Text that only serves to connect ideas without adding new information (e.g., "Furthermore...", "Now we turn to...", "In addition to the above...").
- Author Speculation or Rhetorical Questions: Subjective statements or questions posed to the reader (e.g., "This result is quite surprising.", "But what if the model could...?"). 
- Figures and Tables: Sentences that describe figures and tables.

# Instructions
1.  Read the entire text carefully.
2.  Identify all segments that fit the definition of a "piece of knowledge" and do not fall into the exclusion categories.
3.  Demarcate each piece of knowledge by wrapping it in `<knowledge>` and `</knowledge>` tags.
4.  While you should ensure the knowledge is self-contained and atomic as much as you can, do not just place tags around an entire paragraph. Encapsulate smaller, discrete units of knowledge.
5.  Only return the entire original text with these annotations. Do not modify or summarize the text itself.
"""
# - For instance, it's okay if you have to forgo some portions of a paragraph to break it up into smaller units.
# - Citations: Sentences that primarily exist to cite other work (e.g., "This was previously shown by Smith et al. (2023).").


# Split paper into paragraphs
paper_paragraphs = paper.split('\n\n')

extracted_claims = []
import concurrent.futures

def query_single(paragraph):
    prompt = {}
    prompt['system'] = extraction_prompt
    prompt['user'] = f"""{paragraph}"""
    return utils.query_llm(prompt, model='gpt-4.1', temperature=0, max_tokens=32768)

with concurrent.futures.ThreadPoolExecutor() as executor:
    futures = [executor.submit(query_single, paragraph) for paragraph in paper_paragraphs]
    extracted_claims = [future.result() for future in futures]

# Print each output
for i, output in enumerate(extracted_claims, 1):
    print_wrapped(f"Paragraph {i}: {output}")
    print_wrapped("-" * 50)

Paragraph 1: \title{Direct Preference Optimization: Your Language Model is Secretly a Reward Model}

--------------------------------------------------

Paragraph 2: \begin{abstract} <knowledge>Large-scale unsupervised language models (LMs) learn broad
world knowledge and some reasoning skills.</knowledge> <knowledge>Achieving precise control of the
behavior of large-scale unsupervised LMs is difficult due to the completely unsupervised nature of
their training.</knowledge> <knowledge>Existing methods for gaining steerability in LMs collect
human labels of the relative quality of model generations and fine-tune the unsupervised LM to align
with these preferences, often using reinforcement learning from human feedback (RLHF).</knowledge>
<knowledge>RLHF is a complex and often unstable procedure, which first fits a reward model that
reflects human preferences, and then fine-tunes the large unsupervised LM using reinforcement
learning to maximize this estimated reward without drifting too

### Extracting Knowledge

In [63]:
extraction_prompt = r"""Your task is to act as a text segmenter. Carefully read the provided text from a paper and identify all "pieces of knowledge."

# Definition of a "Piece of Knowledge"
A piece of knowledge is a sentence that provides information. 

# What to Exclude (Do NOT tag these):
- Sentences that can be found in any paper: Sentences that ONLY contain language that's ubiquitous to any paper, adding zero information e.g. "Our results raise several important questions for future work."

# Instructions
1.  Read the entire text carefully.
2.  Identify all sentences that fit the definition of a "piece of knowledge" and do not fall into the exclusion categories.
3.  Please make sure that each table and each figure are tagged as one whole unit. Do not break up the figure or table into smaller units.
4.  For sentences that contain latex, place the tags so that it includes any latex code that's part of the sentence e.g. "\begin{definition}".
5.  Demarcate each piece of knowledge by wrapping it in `<knowledge>` and `</knowledge>` tags.
6.  Please make sure the tags are at the beginning and end of the sentence.
7.  Don't be conservative and include sentences even if it adds little information.
8.  Return the entire original text with these annotations. Do not modify or summarize the text itself.
"""

import re
import pandas as pd

def parse_paper_structure(text):
    """Parse paper into sections, subsections and paragraphs with metadata."""
    sections = []
    
    # Split by sections first
    section_pattern = r'\\section\{([^}]+)\}'
    section_splits = re.split(section_pattern, text)
    
    current_section = "No Section"
    current_section_content = ""
    
    for i in range(len(section_splits)):
        if i == 0:
            # Content before first section
            content = section_splits[i]
            current_section_content = content
        elif i % 2 == 1:
            # This is a section title
            current_section = section_splits[i]
            continue
        else:
            # This is section content
            content = section_splits[i]
            current_section_content = content
        
        # Now split by subsections within this section
        subsection_pattern = r'\\subsection\{([^}]+)\}'
        subsection_splits = re.split(subsection_pattern, content)
        
        current_subsection = "No Subsection"
        current_subsection_content = ""
        
        for j in range(len(subsection_splits)):
            if j == 0:
                # Content before first subsection
                subsection_content = subsection_splits[j]
                current_subsection_content = subsection_content
            elif j % 2 == 1:
                # This is a subsection title
                current_subsection = subsection_splits[j]
                continue
            else:
                # This is subsection content
                subsection_content = subsection_splits[j]
                current_subsection_content = subsection_content
            
            # Split into paragraphs
            paragraphs = [p.strip() for p in subsection_content.split('\n\n') if p.strip()]
            
            for paragraph in paragraphs:
                sections.append({
                    'section': current_section,
                    'subsection': current_subsection,
                    'paragraph': paragraph,
                    'section_text': current_section_content,
                    'subsection_text': current_subsection_content
                })
    
    return pd.DataFrame(sections)

# Parse paper structure
paper_df = parse_paper_structure(paper)

# Process each paragraph with LLM
import concurrent.futures

def query_single(paragraph):
    prompt = {}
    prompt['system'] = extraction_prompt
    prompt['user'] = f"""{paragraph}"""
    return utils.query_llm(prompt, model='gpt-4.1', temperature=0, max_tokens=32768)

with concurrent.futures.ThreadPoolExecutor() as executor:
    futures = [executor.submit(query_single, row['paragraph']) for _, row in paper_df.iterrows()]
    extracted_claims = [future.result() for future in futures]

# Add extracted claims to dataframe
paper_df['extracted_claims'] = extracted_claims

# Print each output with section/subsection context
for i, (_, row) in enumerate(paper_df.iterrows(), 1):
    print_wrapped(f"Section: {row['section']}")
    print_wrapped(f"Subsection: {row['subsection']}")
    print_wrapped(f"Paragraph {i}: {row['extracted_claims']}")
    print_wrapped("-" * 50)

Section: No Section

Subsection: No Subsection

Paragraph 1: <knowledge>\title{Direct Preference Optimization: Your Language Model is Secretly a
Reward Model}</knowledge>

--------------------------------------------------

Section: No Section

Subsection: No Subsection

Paragraph 2: \begin{abstract} <knowledge>While large-scale unsupervised language models (LMs) learn
broad world knowledge and some reasoning skills, achieving precise control of their behavior is
difficult due to the completely unsupervised nature of their training.</knowledge>
<knowledge>Existing methods for gaining such steerability collect human labels of the relative
quality of model generations and fine-tune the unsupervised LM to align with these preferences,
often with reinforcement learning from human feedback (RLHF).</knowledge> <knowledge>However, RLHF
is a complex and often unstable procedure, first fitting a reward model that reflects the human
preferences, and then fine-tuning the large unsupervised LM usi

#### Check Knowledge is actually in the paper

In [78]:
import re
import pandas as pd

def extract_text_from_knowledge_tags(text: str) -> list[str]:
    """
    Finds all <knowledge> tags in a given text and extracts their content.

    Args:
        text: A string containing the text to parse, which may include
              <knowledge>...</knowledge> tags.

    Returns:
        A list of strings, where each string is the content found within
        a <knowledge> tag. The content is stripped of leading/trailing
        whitespace.
    """
    # This regex pattern finds all content between <knowledge> and </knowledge>.
    # - The (.*?) part is a non-greedy capture group for the content inside the tags.
    # - The re.DOTALL flag allows the '.' character to match newlines, so tags
    #   that span multiple lines are correctly handled.
    pattern = re.compile(r'<knowledge>(.*?)</knowledge>', re.DOTALL)
    
    # re.findall returns a list of all captured groups.
    matches = pattern.findall(text)
    
    # Clean up any leading/trailing whitespace from the extracted text.
    cleaned_matches = [match.strip() for match in matches]
    
    return cleaned_matches

def remove_knowledge_tags(text: str) -> str:
    """Remove knowledge tags from text while preserving the content."""
    pattern = re.compile(r'</?knowledge>', re.DOTALL)
    return pattern.sub('', text)

paper_df['extracted_claims'] = extracted_claims

# Extract a list of knowledge statements for each row
paper_df['knowledge_list'] = paper_df['extracted_claims'].apply(extract_text_from_knowledge_tags)

# Explode the DataFrame on the knowledge_list column
paper_df_exploded = paper_df.explode('knowledge_list').rename(columns={'knowledge_list': 'raw_knowledge_statement'})

# Count total claims extracted by the LLM before filtering
total_extracted_claims = paper_df_exploded['raw_knowledge_statement'].notna().sum()

# Filter out claims that are not actually in the original paper text (case-insensitive)
paper_lower = paper.lower()
def is_claim_in_paper(claim):
    # Rows with no knowledge statement (claim is NaN) are kept
    if pd.isna(claim):
        return True
    # Check if the lowercased claim is in the lowercased paper
    return claim.strip().lower() in paper_lower

# Apply the filter and create a new validated dataframe
paper_df_validated = paper_df_exploded[paper_df_exploded['raw_knowledge_statement'].apply(is_claim_in_paper)].copy()

# Count claims that passed validation
validated_claims_count = paper_df_validated['raw_knowledge_statement'].notna().sum()
print(f"Found {validated_claims_count}/{total_extracted_claims} extracted claims in the original paper text.")

# Clean up the original paragraph text by removing knowledge tags from the validated dataframe
paper_df_validated['paragraph'] = paper_df_validated['extracted_claims'].apply(remove_knowledge_tags)

# Drop the now-redundant columns
paper_df_validated = paper_df_validated.drop(columns=['extracted_claims'])

# Display the result
print(f"Total rows after exploding and validation: {len(paper_df_validated)}")
paper_df_validated

Found 217/217 extracted claims in the original paper text.
Total rows after exploding and validation: 222


,section,subsection,paragraph,section_text,subsection_text,raw_knowledge_statement
0,No Section,No Subsection,\title{Direct Preference Optimization: Your La...,\title{Direct Preference Optimization: Your La...,\title{Direct Preference Optimization: Your La...,\title{Direct Preference Optimization: Your La...
1,No Section,No Subsection,\begin{abstract}\nWhile large-scale unsupervis...,\title{Direct Preference Optimization: Your La...,\title{Direct Preference Optimization: Your La...,While large-scale unsupervised language models...
1,No Section,No Subsection,\begin{abstract}\nWhile large-scale unsupervis...,\title{Direct Preference Optimization: Your La...,\title{Direct Preference Optimization: Your La...,Existing methods for gaining such steerability...
1,No Section,No Subsection,\begin{abstract}\nWhile large-scale unsupervis...,\title{Direct Preference Optimization: Your La...,\title{Direct Preference Optimization: Your La...,"However, RLHF is a complex and often unstable ..."
1,No Section,No Subsection,\begin{abstract}\nWhile large-scale unsupervis...,\title{Direct Preference Optimization: Your La...,\title{Direct Preference Optimization: Your La...,In this paper we introduce a new parameterizat...
...,...,...,...,...,...,...
40,Discussion,No Subsection,\textbf{Limitations \& Future Work.} Our resul...,"\nLearning from preferences is a powerful, sca...","\nLearning from preferences is a powerful, sca...","On another front, how does reward over-optimiz..."
40,Discussion,No Subsection,\textbf{Limitations \& Future Work.} Our resul...,"\nLearning from preferences is a powerful, sca...","\nLearning from preferences is a powerful, sca...","Additionally, while we evaluate models up to 6..."
40,Discussion,No Subsection,\textbf{Limitations \& Future Work.} Our resul...,"\nLearning from preferences is a powerful, sca...","\nLearning from preferences is a powerful, sca...","Regarding evaluations, we find that the win ra..."
40,Discussion,No Subsection,\textbf{Limitations \& Future Work.} Our resul...,"\nLearning from preferences is a powerful, sca...","\nLearning from preferences is a powerful, sca...","Finally, many possible applications of DPO exi..."


In [79]:
# Print all the NAs
na_rows = paper_df_validated[paper_df_validated['raw_knowledge_statement'].isna()]
print(f"Found {len(na_rows)} rows with NA knowledge statements:")
for idx, row in na_rows.iterrows():
    print(f"\nRow {idx}:")
    print(f"Paragraph: {row['paragraph']}")

Found 5 rows with NA knowledge statements:

Row 9:
Paragraph: \label{section:prelims}

Row 14:
Paragraph: \label{sec:DPO}

Row 20:
Paragraph: In this section, we give further interpretation of the DPO method, provide theoretical backing, and relate advantages of DPO to issues with actor critic algorithms used for RLHF (such as PPO~\cite{schulman2017proximal}).

Row 21:
Paragraph: \label{sec:theory}

Row 41:
Paragraph: \end{document}


### Now extract all unnecessary latex code from these knowledge statements and call the column "knowledge_statement"

In [80]:

# --- NEW: Clean unnecessary LaTeX from knowledge statements ---
cleaning_prompt = r"""Your task is to remove unmeaningful LaTeX commands while keeping the sentence same otherwise.

# Rules:
1.  **Keep Math**: Preserve mathematical notation written in LaTeX (e.g., `$\mathcal{L}$`, `\pi_\theta`, `\mathbb{E}`). Do not expand or remove them.
2.  **Remove Formatting**: Remove formatting commands like `\textbf{...}`, `\textit{...}`, `\texttt{...}`, etc., but keep the text inside them.
3.  **Handle Citations & References**: Simplify citation commands (e.g., `\cite{...}`, `\citep{...}`) and reference commands (e.g., `\ref{...}`) by removing them but keeping the sentence flow. For example, 'See Appendix~\ref{app:1}' becomes 'See Appendix'.
4.  **Keep Core Content**: The final output must be the original sentence otherwise.
5.  Return only the cleaned sentence, with no extra explanations.

# Example 1:
Input: `Our experiments show that \textbf{DPO} can fine-tune LMs to align with human preferences as well as or better than existing methods.`
Output: `Our experiments show that DPO can fine-tune LMs to align with human preferences as well as or better than existing methods.`

# Example 2:
Input: `The gradient with respect to the parameters $\theta$ can be written as:`
Output: `The gradient with respect to the parameters $\theta$ can be written as:`

# Example 3:
Input: `See Appendix~\ref{app:derivation1} for a complete derivation.`
Output: `See Appendix for a complete derivation.`

# Task:
Now, process the following sentence.
"""

def query_single_clean(statement):
    if statement is None:
        return None
    """Query the LLM to clean a single knowledge statement."""
    prompt = {
        'system': cleaning_prompt,
        'user': f"Input: `{statement}`"
    }
    # Assuming 'gpt-4.1' is a valid model alias in your utils
    return utils.query_llm(prompt, model='gpt-4.1', temperature=0, max_tokens=2048)

paper_df_validated.dropna(inplace=True)
# Get unique non-null statements to avoid redundant API calls
statements_to_clean = paper_df_validated['raw_knowledge_statement'].unique().tolist()
cleaned_statements = []

print(f"Cleaning {len(statements_to_clean)} unique knowledge statements...")
with concurrent.futures.ThreadPoolExecutor(max_workers=16) as executor:
    futures = [executor.submit(query_single_clean, stmt) for stmt in statements_to_clean]
    cleaned_statements = [future.result() for future in futures]

# Create a mapping from raw statement to cleaned statement
cleaning_map = dict(zip(statements_to_clean, cleaned_statements))

# Map the cleaned statements back to the dataframe
paper_df_validated['knowledge_statement'] = paper_df_validated['raw_knowledge_statement'].map(cleaning_map)

paper_df_validated.reset_index(drop=True, inplace=True)
paper_df_validated

Cleaning 217 unique knowledge statements...


,section,subsection,paragraph,section_text,subsection_text,raw_knowledge_statement,knowledge_statement
0,No Section,No Subsection,\title{Direct Preference Optimization: Your La...,\title{Direct Preference Optimization: Your La...,\title{Direct Preference Optimization: Your La...,\title{Direct Preference Optimization: Your La...,Direct Preference Optimization: Your Language ...
1,No Section,No Subsection,\begin{abstract}\nWhile large-scale unsupervis...,\title{Direct Preference Optimization: Your La...,\title{Direct Preference Optimization: Your La...,While large-scale unsupervised language models...,While large-scale unsupervised language models...
2,No Section,No Subsection,\begin{abstract}\nWhile large-scale unsupervis...,\title{Direct Preference Optimization: Your La...,\title{Direct Preference Optimization: Your La...,Existing methods for gaining such steerability...,Existing methods for gaining such steerability...
3,No Section,No Subsection,\begin{abstract}\nWhile large-scale unsupervis...,\title{Direct Preference Optimization: Your La...,\title{Direct Preference Optimization: Your La...,"However, RLHF is a complex and often unstable ...","However, RLHF is a complex and often unstable ..."
4,No Section,No Subsection,\begin{abstract}\nWhile large-scale unsupervis...,\title{Direct Preference Optimization: Your La...,\title{Direct Preference Optimization: Your La...,In this paper we introduce a new parameterizat...,In this paper we introduce a new parameterizat...
...,...,...,...,...,...,...,...
212,Discussion,No Subsection,\textbf{Limitations \& Future Work.} Our resul...,"\nLearning from preferences is a powerful, sca...","\nLearning from preferences is a powerful, sca...","For example, can training with self-labeling f...","For example, can training with self-labeling f..."
213,Discussion,No Subsection,\textbf{Limitations \& Future Work.} Our resul...,"\nLearning from preferences is a powerful, sca...","\nLearning from preferences is a powerful, sca...","On another front, how does reward over-optimiz...","On another front, how does reward over-optimiz..."
214,Discussion,No Subsection,\textbf{Limitations \& Future Work.} Our resul...,"\nLearning from preferences is a powerful, sca...","\nLearning from preferences is a powerful, sca...","Additionally, while we evaluate models up to 6...","Additionally, while we evaluate models up to 6..."
215,Discussion,No Subsection,\textbf{Limitations \& Future Work.} Our resul...,"\nLearning from preferences is a powerful, sca...","\nLearning from preferences is a powerful, sca...","Regarding evaluations, we find that the win ra...","Regarding evaluations, we find that the win ra..."


### Identify Context Required for each Probe

In [81]:
# --- NEW: Add context to knowledge statements ---

context_prompt = r"""Your task is to extract a concise, self-contained piece of information from a larger text. You will be given a `subsection_text` and a `knowledge_statement` that is a direct quote from the text.

Your goal is to expand the `knowledge_statement` by including the minimal necessary preceding context from the `subsection_text` to make it understandable on its own. The output should be a single, contiguous block of text copied "word-for-word" from the `subsection_text`.

# Instructions:
1.  The `knowledge_statement` must be at the end of your output.
2.  The output must be a direct, continuous excerpt from the `subsection_text`. Do not add, remove, or change any words.
3.  Include only the *minimal* amount of preceding text required for the `knowledge_statement` to be clear and self-contained. Avoid including entire paragraphs if a single preceding sentence or phrase is sufficient.
4.  If the `knowledge_statement` is already self-contained, just return the `knowledge_statement` itself.
5.  Return only the final text, with no extra explanations.

# Example:
`subsection_text`: "The Transformer architecture has been very successful. It relies on a self-attention mechanism. This mechanism allows the model to weigh the importance of different words in the input sequence. For example, in the sentence 'The cat sat on the mat', self-attention can help the model understand that 'sat' is related to 'cat' and 'mat'."
`knowledge_statement`: "This mechanism allows the model to weigh the importance of different words in the input sequence."

Output: "The Transformer architecture has been very successful. It relies on a self-attention mechanism. This mechanism allows the model to weigh the importance of different words in the input sequence."

# Task:
Now, process the following.
"""

def query_for_context(subsection_text, raw_knowledge_statement):
    if not isinstance(subsection_text, str) or not isinstance(raw_knowledge_statement, str):
        return None
    prompt = {
        'system': context_prompt,
        'user': f"`subsection_text`:\n{subsection_text}\n\n`knowledge_statement`:\n{raw_knowledge_statement}"
    }
    return utils.query_llm(prompt, model='gpt-4.1', temperature=0, max_tokens=2048)

def check_statement_in_paper(statement):
    if not statement:
        return False
    statement_cleaned = statement.strip().strip('"`')
    return statement_cleaned.lower() in paper.lower()

inputs = [(row['subsection_text'], row['raw_knowledge_statement']) for _, row in paper_df_validated.iterrows()]

print(f"Adding context to {len(inputs)} knowledge statements...")
with concurrent.futures.ThreadPoolExecutor(max_workers=16) as executor:
    futures = [executor.submit(query_for_context, sub_text, raw_stmt) for sub_text, raw_stmt in inputs]
    statements_with_context = [future.result() for future in futures]

paper_df_validated['raw_knowledge_statement_with_context'] = statements_with_context

# Check which statements are found in paper
found_in_paper = [check_statement_in_paper(stmt) for stmt in statements_with_context]

# Retry failed statements twice
for retry in range(2):
    failed_indices = [i for i, found in enumerate(found_in_paper) if not found]
    if not failed_indices:
        break
    
    print(f"Retry {retry + 1}: Processing {len(failed_indices)} failed statements...")
    retry_inputs = [inputs[i] for i in failed_indices]
    
    with concurrent.futures.ThreadPoolExecutor(max_workers=16) as executor:
        futures = [executor.submit(query_for_context, sub_text, raw_stmt) for sub_text, raw_stmt in retry_inputs]
        retry_results = [future.result() for future in futures]
    
    for i, result in zip(failed_indices, retry_results):
        statements_with_context[i] = result
        found_in_paper[i] = check_statement_in_paper(result)

# Update dataframe with final results
paper_df_validated['raw_knowledge_statement_with_context'] = statements_with_context

# Add context_needed column
paper_df_validated['context_needed'] = [
    stmt_with_context.strip() != raw_stmt.strip() 
    for stmt_with_context, raw_stmt in zip(statements_with_context, paper_df_validated['raw_knowledge_statement'])
]

# Filter out statements not found in paper
initial_rows = len(paper_df_validated)
paper_df_validated = paper_df_validated[found_in_paper]
final_rows = len(paper_df_validated)

print(f"Filtered out {initial_rows - final_rows} rows that were not found in the original paper text.")
print(f"Remaining rows: {final_rows}")

paper_df_validated

Adding context to 217 knowledge statements...
Retry 1: Processing 4 failed statements...
Retry 2: Processing 3 failed statements...
Filtered out 2 rows that were not found in the original paper text.
Remaining rows: 215


,section,subsection,paragraph,section_text,subsection_text,raw_knowledge_statement,knowledge_statement,raw_knowledge_statement_with_context,context_needed
0,No Section,No Subsection,\title{Direct Preference Optimization: Your La...,\title{Direct Preference Optimization: Your La...,\title{Direct Preference Optimization: Your La...,\title{Direct Preference Optimization: Your La...,Direct Preference Optimization: Your Language ...,\title{Direct Preference Optimization: Your La...,False
1,No Section,No Subsection,\begin{abstract}\nWhile large-scale unsupervis...,\title{Direct Preference Optimization: Your La...,\title{Direct Preference Optimization: Your La...,While large-scale unsupervised language models...,While large-scale unsupervised language models...,While large-scale unsupervised language models...,False
2,No Section,No Subsection,\begin{abstract}\nWhile large-scale unsupervis...,\title{Direct Preference Optimization: Your La...,\title{Direct Preference Optimization: Your La...,Existing methods for gaining such steerability...,Existing methods for gaining such steerability...,While large-scale unsupervised language models...,True
3,No Section,No Subsection,\begin{abstract}\nWhile large-scale unsupervis...,\title{Direct Preference Optimization: Your La...,\title{Direct Preference Optimization: Your La...,"However, RLHF is a complex and often unstable ...","However, RLHF is a complex and often unstable ...",Existing methods for gaining such steerability...,True
4,No Section,No Subsection,\begin{abstract}\nWhile large-scale unsupervis...,\title{Direct Preference Optimization: Your La...,\title{Direct Preference Optimization: Your La...,In this paper we introduce a new parameterizat...,In this paper we introduce a new parameterizat...,"However, RLHF is a complex and often unstable ...",True
...,...,...,...,...,...,...,...,...,...
212,Discussion,No Subsection,\textbf{Limitations \& Future Work.} Our resul...,"\nLearning from preferences is a powerful, sca...","\nLearning from preferences is a powerful, sca...","For example, can training with self-labeling f...","For example, can training with self-labeling f...",Our initial results suggest that DPO policies ...,True
213,Discussion,No Subsection,\textbf{Limitations \& Future Work.} Our resul...,"\nLearning from preferences is a powerful, sca...","\nLearning from preferences is a powerful, sca...","On another front, how does reward over-optimiz...","On another front, how does reward over-optimiz...",Our initial results suggest that DPO policies ...,True
214,Discussion,No Subsection,\textbf{Limitations \& Future Work.} Our resul...,"\nLearning from preferences is a powerful, sca...","\nLearning from preferences is a powerful, sca...","Additionally, while we evaluate models up to 6...","Additionally, while we evaluate models up to 6...","Additionally, while we evaluate models up to 6...",False
215,Discussion,No Subsection,\textbf{Limitations \& Future Work.} Our resul...,"\nLearning from preferences is a powerful, sca...","\nLearning from preferences is a powerful, sca...","Regarding evaluations, we find that the win ra...","Regarding evaluations, we find that the win ra...","Additionally, while we evaluate models up to 6...",True


In [82]:
# print out the rows where context_needed is True. print the raw_knowledge_statement and the raw_knowledge_statement_with_context
context_needed_rows = paper_df_validated[paper_df_validated['context_needed'] == True]

for idx, row in context_needed_rows.iterrows():
    print(f"Row {idx}:")
    print(f"Raw statement: {row['raw_knowledge_statement']}")
    print(f"Statement with context: {row['raw_knowledge_statement_with_context']}")
    print("-" * 80)
    

Row 2:
Raw statement: Existing methods for gaining such steerability collect human labels of the relative quality of model generations and fine-tune the unsupervised LM to align with these preferences, often with reinforcement learning from human feedback (RLHF).
Statement with context: While large-scale unsupervised language models (LMs) learn broad world knowledge and some reasoning skills, achieving precise control of their behavior is difficult due to the completely unsupervised nature of their training.
Existing methods for gaining such steerability collect human labels of the relative quality of model generations and fine-tune the unsupervised LM to align with these preferences, often with reinforcement learning from human feedback (RLHF).
--------------------------------------------------------------------------------
Row 3:
Raw statement: However, RLHF is a complex and often unstable procedure, first fitting a reward model that reflects the human preferences, and then fine-tuni

In [83]:
# save the paper_df_validated to a csv file
paper_df_validated.to_csv('DPO_knowledge_probes.csv', index=False)

### Identify Target Span Given the Piece of Knowledge

In [ ]:
pass

### Creating Ontologies for the Knowledge

In [ ]:
for claim in extracted_claims:
    prompt = {}

    prompt['system'] = """Your task is to act as a strict claim evaluator. You will be given a single claim. Your goal is to determine if this claim meets ALL of the following criteria.

    # Evaluation Criteria

    1.  **Objective and Factual:** Extract statements presented as facts or claims, not opinions, subjective evaluations, or superlative statements.
        *   GOOD: "Direct Preference Optimization (DPO) is an algorithm for aligning language models with human preferences."
        *   BAD: "We believe DPO is the most promising approach for alignment." (Subjective belief)
        *   BAD: "DPO is the best method." (Superlative statement)

    3.  **Generalizable Knowledge, not Paper-Specific Results:** Focus on definitions, mechanisms, and core concepts. Do not extract claims about the paper's specific findings, experimental setup, or citations.
        *   GOOD: "Reinforcement Learning from Human Feedback (RLHF) typically involves training a separate reward model on preference data."
        *   BAD: "Our experiments show a 5% improvement on the benchmark." (Specific result)
        *   BAD: "As shown by Smith et al. (2023), the method is effective." (Relies on a citation)"""

    prompt['user'] = f"""Claim: {claim}. Does this claim strictly meet all four criteria? Respond with only the word "True" or "False"."""
    output = utils.query_llm(prompt, model='gpt-4.1', max_tokens=2)
    print(output)